In [ ]:
import os
from dotenv import load_dotenv
from mwrogue.esports_client import EsportsClient
from mwrogue.auth_credentials import AuthCredentials

load_dotenv()

credentials = AuthCredentials(
    username=os.getenv("LEAGUEPEDIA_USER"),
    password=os.getenv("LEAGUEPEDIA_PASS"),
)
site = EsportsClient("lol", credentials=credentials)

In [ ]:
print(load_dotenv())                   # True means the .env file was found
#print(os.getenv("LEAGUEPEDIA_USER"))   # should show YourName@yourbot

info = site.client.api("query", meta="userinfo", uiprop="groups")
print("anon" not in info["query"]["userinfo"])   # True = logged in

In [ ]:
import duckdb
import time

page = "LCK/2026 Season/Rounds 1-2"

con = duckdb.connect("lol_rankings.duckdb")
con.execute(open("docs/raw_schema.sql").read())

In [ ]:
import html

def clean(v):
    if isinstance(v, list):
        v = ",".join(v)
    return None if v in (None, "") else html.unescape(v)

In [ ]:
def sql_list(names):
    return ", ".join("'" + n.replace("'", "\\'") + "'" for n in names)

In [ ]:
def pull_full(cargo_table, raw_table, cols, upsert=False):
    rows = site.cargo_client.query(tables=cargo_table, fields=", ".join(cols))
    values = [[clean(r.get(c, r.get(c.replace("_", " ")))) for c in cols] for r in rows]
    if not values:
        raise ValueError(f"No rows for {cargo_table}")

    verb = "INSERT OR REPLACE" if upsert else "INSERT"
    col_names = ", ".join(cols)
    placeholders = ", ".join("?" for _ in cols)

    con.execute("BEGIN")
    con.execute(f"DELETE FROM raw.{raw_table}")
    con.executemany(f"{verb} INTO raw.{raw_table} ({col_names}) VALUES ({placeholders})", values)
    con.execute("INSERT INTO raw.pull_log VALUES (?, ?, now(), ?)", [raw_table, "all", len(values)])
    con.execute("COMMIT")

    time.sleep(5)
    return len(values)

In [ ]:
def already_pulled(raw_table, page):
    count = con.execute("""
        SELECT COUNT(*) FROM raw.pull_log
        WHERE table_name = ? AND scope = ?
    """, [raw_table, page]).fetchone()[0]
    return count > 0

In [ ]:
def pull_table(cargo_table, raw_table, cols, page, page_col="OverviewPage", allow_empty=False, force=False):
    if not force and already_pulled(raw_table, page):
        print(f"  skip {raw_table}: already pulled")
        return 0

    where = f"{page_col} = '{page}'"

    rows = site.cargo_client.query(tables=cargo_table, fields=", ".join(cols), where=where)

    values = [[clean(r.get(c, r.get(c.replace("_", " ")))) for c in cols] for r in rows]

    if not values:
        if allow_empty:
            return 0
        raise ValueError(f"No rows for {cargo_table} / {page}")

    col_names = ", ".join(cols)
    placeholders = ", ".join("?" for _ in cols)

    con.execute("BEGIN")
    con.execute(f"DELETE FROM raw.{raw_table} WHERE {page_col} = ?", [page])
    con.executemany(f"INSERT INTO raw.{raw_table} ({col_names}) VALUES ({placeholders})", values)
    con.execute("INSERT INTO raw.pull_log VALUES (?, ?, now(), ?)", [raw_table, page, len(values)])
    con.execute("COMMIT")

    time.sleep(5)
    return len(values)

In [ ]:
games_cols = ["GameId", "MatchId", "OverviewPage", "Tournament", "N_GameInMatch", "DateTime_UTC",
              "Team1", "Team2", "WinTeam", "LossTeam", "Winner", "Gamelength_Number",
              "Patch", "PatchSort", "RiotPlatformGameId"]
print(pull_table("ScoreboardGames", "scoreboard_games", games_cols, page))   # 204

In [ ]:
print(con.execute("""
    SELECT COUNT(*), COUNT(DISTINCT GameId)
    FROM raw.scoreboard_games
    WHERE OverviewPage = ?
""", [page]).fetchone())

In [ ]:
teams_cols = ["GameTeamId", "GameId", "MatchId", "OverviewPage", "Team", "Side", "Number", "IsWinner",
              "Bans", "Picks", "Roster", "Dragons", "Clouds", "Infernals", "Mountains", "Oceans",
              "Hextechs", "Chemtechs", "Elders", "Barons", "Towers", "Gold", "Kills",
              "RiftHeralds", "VoidGrubs", "Atakhans", "Inhibitors"]
print(pull_table("ScoreboardTeams", "scoreboard_teams", teams_cols, page))   # 408

In [ ]:
print(con.execute("""
    SELECT COUNT(*) FROM raw.scoreboard_teams
    WHERE OverviewPage = ? AND IsWinner
""", [page]).fetchone())

In [ ]:
players_cols = ["GameRoleId", "GameId", "GameTeamId", "MatchId", "OverviewPage", "Link", "Name",
                "Team", "TeamVs", "Role", "Role_Number", "Side", "Champion", "Kills", "Deaths",
                "Assists", "Gold", "CS", "DamageToChampions", "VisionScore", "TeamKills",
                "TeamGold", "PlayerWin", "DateTime_UTC"]
print(pull_table("ScoreboardPlayers", "scoreboard_players", players_cols, page))   # 2040

In [ ]:
print(con.execute("""
    SELECT COUNT(*)
    FROM raw.scoreboard_players p
    JOIN raw.scoreboard_teams t USING (GameTeamId)
    JOIN raw.scoreboard_games g ON t.GameId = g.GameId
    WHERE p.OverviewPage = ?
""", [page]).fetchone())

In [ ]:
match_cols = ["MatchId", "OverviewPage", "Team1", "Team2", "Winner", "Team1Score", "Team2Score",
              "FF", "IsNullified", "IsTiebreaker", "BestOf", "DateTime_UTC", "HasTime", "Tab",
              "Round", "Phase", "GroupName", "MatchDay", "Patch", "DisabledChampions"]
print(pull_table("MatchSchedule", "match_schedule", match_cols, page))   # 90

In [ ]:
print(con.execute("""
    SELECT SUM(Team1Score + Team2Score) FROM raw.match_schedule WHERE OverviewPage = ?
""", [page]).fetchone())

In [ ]:
print(con.execute("SELECT COUNT(*), COUNT(Patch), COUNT(N_GameInMatch) FROM raw.scoreboard_games").fetchone())

In [ ]:
match_game_cols = ["GameId", "MatchId", "OverviewPage", "N_GameInMatch", "Blue", "Red", "Winner",
                   "IsRemake", "IsChronobreak", "FF", "Selection", "PickSelection", "FirstPick",
                   "FirstSelection", "HasSelection", "MVP", "RiotPlatformGameId"]
print(pull_table("MatchScheduleGame", "match_schedule_game", match_game_cols, page))   # 204

In [ ]:
pb_cols = ["GameId", "MatchId", "OverviewPage", "N_GameInMatch", "Team1", "Team2", "Winner", "SwitchedSides"]
for kind in ["Ban", "Pick", "Role"]:
    for team in [1, 2]:
        pb_cols += [f"Team{team}{kind}{i}" for i in range(1, 6)]
pb_cols += ["Team1PicksByRoleOrder", "Team2PicksByRoleOrder", "IsComplete"]
print(pull_table("PicksAndBansS7", "picks_and_bans", pb_cols, page))   # 204

In [ ]:
print(con.execute("""
    SELECT COALESCE(SwitchedSides, false), COUNT(*), SUM((p.Team1 <> m.Blue)::INT)
    FROM raw.picks_and_bans p JOIN raw.match_schedule_game m USING (GameId)
    GROUP BY 1 ORDER BY 1
""").fetchall())   # [(False, 62, 0), (True, 142, 142)]

In [ ]:
postgame_cols = ["GameId", "OverviewPage", "RiotPlatformGameId", "StatsPage", "TimelinePage"]
print(pull_table("PostgameJsonMetadata", "postgame_json_metadata", postgame_cols, page))   # 204

In [ ]:
tournament_players_cols = ["OverviewPage", "Team", "Link", "Player", "Role", "N_PlayerInTeam"]
print(pull_table("TournamentPlayers", "tournament_players", tournament_players_cols, page))   # players + coaches

In [ ]:
tournament_cols = ["OverviewPage", "Name", "StandardName", "League", "Region", "Year", "Split",
                   "SplitNumber", "DateStart", "Date", "TournamentLevel", "IsQualifier",
                   "IsPlayoffs", "IsOfficial", "EventType", "Organizer", "Country", "BasePage",
                   "AlternativeNames"]
print(pull_table("Tournaments", "tournaments", tournament_cols, page))   # 1

In [ ]:
results_cols = ["OverviewPage", "RosterPage", "Event", "Tier", "Date", "Team", "Place",
                "Place_Number", "Phase", "Qualified", "Prize_USD", "IsAchievement", "Showmatch"]
print(pull_table("TournamentResults", "tournament_results", results_cols, page, page_col="RosterPage"))

In [ ]:
awards_cols = ["Player", "Link", "Team", "OverviewPage", "Display", "Place", "Place_Number", "AchievementType"]
print(pull_table("IndividualAchievements", "individual_achievements", awards_cols,
                 "LCK/2026 Season/Rounds 3-4", allow_empty=True))   # 19

In [ ]:
print(pull_full("Leagues", "leagues", ["League", "League_Short", "Region", "Level", "IsOfficial"], upsert=True))   # ~194
print(pull_full("Champions", "champions", ["Name", "Attributes", "Resource", "ReleaseDate"], upsert=True))        # ~173
print(pull_full("TeamRenames", "team_renames", ["Date", "OriginalName", "NewName", "Verb", "Slot", "IsSamePage", "NewsId"]))   # ~1,256
print(pull_full("PlayerRenames", "player_renames", ["Date", "OriginalName", "NewName", "IsRestyle", "NewsId"]))                # ~2,556

In [ ]:
def pull_by_names(cargo_table, raw_table, cols, key_col, names, upsert=False):
    verb = "INSERT OR REPLACE" if upsert else "INSERT"
    col_names = ", ".join(cols)
    placeholders = ", ".join("?" for _ in cols)
    total = 0

    for i in range(0, len(names), 50):
        chunk = names[i:i + 50]
        rows = site.cargo_client.query(tables=cargo_table, fields=", ".join(cols),
                                       where=f"{key_col} IN ({sql_list(chunk)})")
        values = [[clean(r.get(c, r.get(c.replace("_", " ")))) for c in cols] for r in rows]
        time.sleep(5)
        if not values:
            continue

        marks = ", ".join("?" for _ in chunk)
        con.execute("BEGIN")
        con.execute(f"DELETE FROM raw.{raw_table} WHERE lower({key_col}) IN ({marks})", [n.lower() for n in chunk])
        con.executemany(f"{verb} INTO raw.{raw_table} ({col_names}) VALUES ({placeholders})", values)
        con.execute("INSERT INTO raw.pull_log VALUES (?, ?, now(), ?)", [raw_table, f"{len(chunk)} names", len(values)])
        con.execute("COMMIT")
        total += len(values)

    return total

In [ ]:
people = sorted({r[0] for r in con.execute("""
    SELECT upper(left(Link, 1)) || substr(Link, 2) FROM raw.scoreboard_players WHERE Link IS NOT NULL
    UNION
    SELECT upper(left(Link, 1)) || substr(Link, 2) FROM raw.tournament_players WHERE Link IS NOT NULL
""").fetchall()})
print(len(people))   # roughly 100: players + coaches

In [ ]:
players_ref_cols = ["OverviewPage", "ID", "Player", "Name", "NativeName", "Country", "NationalityPrimary",
                    "Nationality", "Birthdate", "Residency", "ResidencyFormer", "Role", "IsRetired",
                    "IsSubstitute", "IsTrainee", "IsPersonality"]
print(pull_by_names("Players", "players", players_ref_cols, "OverviewPage", people, upsert=True))   # about len(people)

print(pull_by_names("PlayerRedirects", "player_redirects", ["AllName", "OverviewPage", "ID"],
                    "OverviewPage", people, upsert=True))   # several per player

print(pull_by_names("Tenures", "tenures", ["Player", "Team", "DateJoin", "DateLeave", "ContractEnd",
                    "RosterChangeIdJoin", "RosterChangeIdLeave", "NextTeam", "NextIsRetired", "IsCurrent"],
                    "Player", people))   # hundreds

print(pull_by_names("RosterChanges", "roster_changes", ["RosterChangeId", "Date_Sort", "Player", "Direction",
                    "Team", "RolesIngame", "RolesStaff", "Role", "Status", "Tournaments", "NewsId"],
                    "Player", people))   # more than tenures

In [ ]:
team_names = sorted({r[0] for r in con.execute("""
    SELECT Team FROM raw.scoreboard_teams
    UNION SELECT Team1 FROM raw.match_schedule
    UNION SELECT Team2 FROM raw.match_schedule
    UNION SELECT Team FROM raw.tournament_players
    UNION SELECT Team FROM raw.tournament_results
    UNION SELECT Team FROM raw.tenures
    UNION SELECT Team FROM raw.roster_changes
""").fetchall() if r[0]})
print(len(team_names))

In [ ]:
for i in range(0, len(team_names), 50):
    chunk = team_names[i:i + 50]
    rows = site.cargo_client.query(tables="TeamRedirects", fields="AllName, _pageName=Target",
                                   where=f"AllName IN ({sql_list(chunk)})")
    values = [[clean(r["AllName"]), clean(r["Target"])] for r in rows]
    time.sleep(5)
    if values:
        con.executemany("INSERT OR REPLACE INTO raw.team_redirects (AllName, _pageName) VALUES (?, ?)", values)

print(con.execute("SELECT COUNT(*), COUNT(_pageName) FROM raw.team_redirects").fetchone())   # both numbers equal

In [ ]:
targets = {r[0] for r in con.execute("SELECT DISTINCT _pageName FROM raw.team_redirects").fetchall() if r[0]}
team_pages = sorted(set(team_names) | targets)

teams_ref_cols = ["OverviewPage", "Name", "Short", "Region", "Location", "OrganizationPage", "IsDisbanded", "RenamedTo"]
print(pull_by_names("Teams", "teams", teams_ref_cols, "OverviewPage", team_pages, upsert=True))

In [ ]:
print(len(people))

In [ ]:
print(con.execute("""
    WITH names AS (SELECT DISTINCT Team AS name FROM raw.scoreboard_teams WHERE OverviewPage = ?),
    resolved AS (
        SELECT n.name, COALESCE(r._pageName, n.name) AS team_page
        FROM names n LEFT JOIN raw.team_redirects r ON r.AllName = n.name
    )
    SELECT name FROM resolved
    WHERE team_page NOT IN (SELECT OverviewPage FROM raw.teams)
""", [page]).fetchall())   # []

In [ ]:
print(con.execute("""
    WITH links AS (
        SELECT DISTINCT Link, upper(left(Link, 1)) || substr(Link, 2) AS fixed
        FROM raw.scoreboard_players WHERE OverviewPage = ?
    ),
    resolved AS (
        SELECT l.Link, COALESCE(r.OverviewPage, l.fixed) AS player_page
        FROM links l LEFT JOIN raw.player_redirects r ON r.AllName = l.fixed
    )
    SELECT Link FROM resolved
    WHERE player_page NOT IN (SELECT OverviewPage FROM raw.players)
""", [page]).fetchall())   # [] or one or two names

In [ ]:
print(con.execute("""
    SELECT COUNT(DISTINCT tp.Link), COUNT(DISTINCT t.Player)
    FROM raw.tournament_players tp
    LEFT JOIN raw.tenures t ON t.Player = upper(left(tp.Link, 1)) || substr(tp.Link, 2)
    WHERE tp.OverviewPage = ? AND tp.Role = 'Coach'
""", [page]).fetchone())   # both numbers equal

In [ ]:
print(con.execute("""
    WITH names AS (SELECT DISTINCT Team AS name FROM raw.tenures WHERE Team IS NOT NULL),
    resolved AS (
        SELECT n.name, COALESCE(r._pageName, n.name) AS team_page
        FROM names n LEFT JOIN raw.team_redirects r ON lower(r.AllName) = lower(n.name)
    )
    SELECT COUNT(*),
           COUNT(*) FILTER (WHERE lower(team_page) IN (SELECT lower(OverviewPage) FROM raw.teams))
    FROM resolved
""").fetchone())

In [ ]:
for row in con.execute("""
    WITH names AS (SELECT DISTINCT Team AS name FROM raw.tenures WHERE Team IS NOT NULL),
    resolved AS (
        SELECT n.name, r._pageName AS target
        FROM names n LEFT JOIN raw.team_redirects r ON r.AllName = n.name
    )
    SELECT name, target FROM resolved
    WHERE COALESCE(target, name) NOT IN (SELECT OverviewPage FROM raw.teams)
    ORDER BY name
""").fetchall():
    print(row)

In [ ]:
tier1_leagues = [
    "LoL Champions Korea",
    "Tencent LoL Pro League",
    "LoL EMEA Championship",
    "League of Legends Championship Series",
    "League of Legends Championship Pacific",
    "Circuit Brazilian League of Legends",
]

rows = site.cargo_client.query(
    tables="Tournaments",
    fields="OverviewPage, League, DateStart, IsPlayoffs",
    where=f"Year = '2026' AND TournamentLevel = 'Primary' AND League IN ({sql_list(tier1_leagues)})",
    order_by="DateStart",
)
time.sleep(5)

for r in rows:
    print(r["OverviewPage"], r["DateStart"])
print(len(rows))   # about 30-35

In [ ]:
print(len(rows))   # about 30-35

pages = [r["OverviewPage"] for r in rows
         if "Showmatch" not in r["OverviewPage"] and "Promotion" not in r["OverviewPage"]]

pages += [
    "2026 First Stand",                            # 45 games
    "2026 Mid-Season Invitational",                # 71 games
    "LPL/2026 Season/Regional Finals",             # LPL's Worlds qualifier
    "2026 Season World Championship/Play-In",      # 22 games
    "2026 Season World Championship/Main Event",   # 58 so far, still being played
]
print(len(pages))   # 38

In [ ]:
def pull_tournament(page, force=False):
    counts = {}
    counts["games"] = pull_table("ScoreboardGames", "scoreboard_games", games_cols, page, force=force)
    counts["teams"] = pull_table("ScoreboardTeams", "scoreboard_teams", teams_cols, page, force=force)
    counts["players"] = pull_table("ScoreboardPlayers", "scoreboard_players", players_cols, page, force=force)
    counts["series"] = pull_table("MatchSchedule", "match_schedule", match_cols, page, force=force)
    counts["game_sides"] = pull_table("MatchScheduleGame", "match_schedule_game", match_game_cols, page, force=force)
    counts["drafts"] = pull_table("PicksAndBansS7", "picks_and_bans", pb_cols, page, force=force, allow_empty=True)
    counts["timelines"] = pull_table("PostgameJsonMetadata", "postgame_json_metadata", postgame_cols, page, force=force, allow_empty=True)
    counts["rosters"] = pull_table("TournamentPlayers", "tournament_players", tournament_players_cols, page, force=force, allow_empty=True)
    counts["tournament"] = pull_table("Tournaments", "tournaments", tournament_cols, page, force=force)
    counts["results"] = pull_table("TournamentResults", "tournament_results", results_cols, page,
                                   page_col="RosterPage", force=force, allow_empty=True)
    counts["awards"] = pull_table("IndividualAchievements", "individual_achievements", awards_cols, page,
                                  force=force, allow_empty=True)
    return counts

In [ ]:
def check_tournament(page):
    q = lambda sql: con.execute(sql, [page]).fetchone()[0]
    games = q("SELECT COUNT(*) FROM raw.scoreboard_games WHERE OverviewPage = ?")
    expected = q("SELECT SUM(Team1Score + Team2Score) FROM raw.match_schedule WHERE OverviewPage = ? AND COALESCE(FF, 0) = 0")
    print(page)
    print("  games vs series scores:", games, expected)
    print("  team rows (2x games):  ", q("SELECT COUNT(*) FROM raw.scoreboard_teams WHERE OverviewPage = ?"))
    print("  player rows (10x games):", q("SELECT COUNT(*) FROM raw.scoreboard_players WHERE OverviewPage = ?"))
    print("  timelines:             ", q("SELECT COUNT(TimelinePage) FROM raw.postgame_json_metadata WHERE OverviewPage = ?"))

In [ ]:
failed = []
for p in pages:
    force = p.startswith("2026 Season World Championship")   # still being played
    try:
        pull_tournament(p, force=force)
        check_tournament(p)
    except Exception as e:
        print(f"FAILED {p}: {e}")
        failed.append(p)
print("failed:", failed)

In [ ]:
con.close()